# Solutions to Exercises: Extending a Knowledge Graph

In these exercises we are going to get some practice in working with a OWL graph, implementing a simple extension of the graph we created in the lecture.
If you have not already done so, please run the full notebook 2.07 and save out the graph using the final cell.

Now run the following cell:

In [ ]:
from owlready2 import *
onto = get_ontology("teaching.owl").load()

x = list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?student
    {
        ?student rdf:type RDF:Student
    }
    """))
for i in x:
    print(i[0].person_name[0])

If this has printed a list of all the students then we are good to continue.

1. Extend the ontology to include a new relation `Advises({Professor},{Student})`. [Hint: you will need to use the prefix `onto` when specifying the domain and range of the relation].

In [ ]:
with onto:
    class Advises(ObjectProperty):
        domain = [onto.Professor]
        range = [onto.Student]

2. Write a SPARQL query to find the list of Professors and the students that they teach (ie. `Advises(Professor, Student)`).

In [ ]:
AdvisorList = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
        SELECT ?staff ?student
        WHERE{
            ?staff rdf:type ONTO:Professor
            ?student rdf:type ONTO:Student
            ?module rdf:type ONTO:Module
            ?course rdf:type ONTO:Course
            ?staff ONTO:Teaches ?module
            ?course ONTO:Requires ?module
            ?student ONTO:Enrolled ?course            
        }
        """))
print(len(AdvisorList))
for i in AdvisorList:
    print(i[0].person_name[0] + " advises " + i[1].person_name[0])

3. Populate the graph with the results of the SPARQL Query.

In [ ]:
for i in AdvisorList:
    getattr(i[0],'Advises').append(i[1])

4. Write a SPARQL query that uses the `Advises` relationship to perform the same query as in part 2. 

Are the results the same? If not, why?

In [ ]:
AdvisorList = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
        SELECT ?staff ?student
        WHERE{
            ?staff rdf:type ONTO:Professor
            ?student rdf:type ONTO:Student
            ?staff ONTO:Advises ?student            
        }
        """))
print(len(AdvisorList))
for i in AdvisorList:
    print(i[0].person_name[0] + " advises " + i[1].person_name[0])

The results are not quite the same because there are some members of staff who teach the same students on more than one module. The original query returns every instance, whereas the `Advises` relation in the graph is created only once.